In [34]:
%pip install nltk
%pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# NLP PIPELINE USED
CSV → Clean → Remove Nulls → Convert Rating → Create Sentiment
→ Remove Neutral → Clean Text → Split Data

In [35]:

# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)


# 2. DOWNLOAD NLTK DATA

nltk.download('stopwords')
nltk.download('wordnet')

#  3. LOAD DATASET (AMAZON REVIEWS)

df = pd.read_csv("Amazon_Reviews.csv") 

# Keep required columns
df = df[['Review Text', 'Rating']]

# Drop null values
df = df.dropna()

print("Initial Shape:", df.shape)

# 4. FIX RATING COLUMN

df['Rating'] = df['Rating'].astype(str).apply(
    lambda x: re.findall(r'\d+', x)[0] if re.findall(r'\d+', x) else None
)

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')


#  5. CREATE SENTIMENT LABELS

def get_sentiment(rating):
    if rating >= 4:
        return 1   # Positive
    elif rating <= 2:
        return 0   # Negative
    else:
        return None  # Neutral

df['sentiment'] = df['Rating'].apply(get_sentiment)

print("\nSentiment BEFORE drop:")
print(df['sentiment'].value_counts(dropna=False))

# Remove neutral values
df = df.dropna(subset=['sentiment'])

print("\nShape AFTER drop:", df.shape)


#  6. TEXT PREPROCESSING

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

df['clean_review'] = df['Review Text'].apply(clean_text)


#  7. TRAIN-TEST SPLIT

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTrain Size:", len(X_train))
print("Test Size:", len(X_test))

#  8. FEATURE ENGINEERING

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)


#  9. MODEL TRAINING

lr = LogisticRegression(max_iter=200)
nb = MultinomialNB()
dt = DecisionTreeClassifier()

lr.fit(X_train_tfidf, y_train)
nb.fit(X_train_bow, y_train)
dt.fit(X_train_bow, y_train)


#  10. PREDICTIONS

y_pred_lr = lr.predict(X_test_tfidf)
y_pred_nb = nb.predict(X_test_bow)
y_pred_dt = dt.predict(X_test_bow)


#  11. EVALUATION FUNCTION

def evaluate(y_test, y_pred, model_name):
    print(f"\n📊 {model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Evaluate models
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")


# 🔲 12. CONFUSION MATRICES (TEXT FORMAT)

print("\nConfusion Matrix - Logistic Regression:\n", confusion_matrix(y_test, y_pred_lr))
print("\nConfusion Matrix - Naive Bayes:\n", confusion_matrix(y_test, y_pred_nb))
print("\nConfusion Matrix - Decision Tree:\n", confusion_matrix(y_test, y_pred_dt))


# 📊 13. MODEL COMPARISON TABLE

models = ['Logistic Regression', 'Naive Bayes', 'Decision Tree']

accuracies = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_dt)
]

f1_scores = [
    f1_score(y_test, y_pred_lr),
    f1_score(y_test, y_pred_nb),
    f1_score(y_test, y_pred_dt)
]

results = pd.DataFrame({
    'Model': models,
    'Accuracy': accuracies,
    'F1 Score': f1_scores
})

print("\nModel Comparison:\n")
print(results.sort_values(by='F1 Score', ascending=False))


#  14. BEST MODEL

best_model = models[np.argmax(accuracies)]
print("\n🏆 Best Model:", best_model)





[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Initial Shape: (21055, 2)

Sentiment BEFORE drop:
sentiment
0.0    14350
1.0     5820
NaN      885
Name: count, dtype: int64

Shape AFTER drop: (20170, 3)

Train Size: 16136
Test Size: 4034

📊 Logistic Regression
Accuracy: 0.9295984134853743
Precision: 0.9182509505703422
Recall: 0.8298969072164949
F1 Score: 0.871841155234657

Classification Report:
               precision    recall  f1-score   support

         0.0       0.93      0.97      0.95      2870
         1.0       0.92      0.83      0.87      1164

    accuracy                           0.93      4034
   macro avg       0.93      0.90      0.91      4034
weighted avg       0.93      0.93      0.93      4034


📊 Naive Bayes
Accuracy: 0.917203767972236
Precision: 0.8304140127388535
Recall: 0.8960481099656358
F1 Score: 0.8619834710743801

Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.93      0.94      2870
         1.0       0.83      0.90      0.86      1164

   

# Comparison & Insights
TF-IDF with Logistic Regression performed best because it captures word importance and handles high-dimensional text data efficiently, while proper preprocessing reduced noise and improved accuracy.